# AquaSentinel AI - accuracy improvement experiment

This notebook keeps the current best.pt baseline intact and trains a separate candidate. The supplied plots suggest the first YOLOv8n run plateaued around precision 0.65, recall 0.51, mAP50 0.51, and mAP50-95 0.37 (approximate values read from the chart). The confusion matrix suggests weaker recall for Fragment and Microbeads and many background-column predictions. Review validation predictions before choosing a new model.

Workflow: audit labels; optionally add the training split of a licensed wastewater microscopy dataset as Fragment-only supplemental training data; validate the current checkpoint; train a YOLOv8s candidate at 960 pixels with early stopping; compare validation results and confidence thresholds; save the candidate to Drive; then evaluate the held-out test split. The external dataset is not used in validation or test.

Higher resolution, a larger model, and supplemental data are experiments, not guarantees of improvement. The external wastewater images come from a different imaging pipeline, so review sample predictions and compare validation performance. Detections are image-based screening of suspected particles, not chemical identification.


In [ ]:
!pip -q install "ultralytics>=8.3,<9" "datasets>=3,<5"


In [ ]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
from collections import Counter
import yaml, json, shutil, torch
from ultralytics import YOLO

drive.mount('/content/drive')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Runtime > Change runtime type, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))

## Dataset location

If the dataset is already extracted in /content/microplastic_dataset, the notebook reuses it. Otherwise, the next cell asks you to select the dataset ZIP from your computer. That is a dataset upload through a code cell. Do not choose the ZIP with File > Upload notebook.

In [ ]:
# Colab can use the dataset already stored in your Drive, avoiding a new ZIP upload.
DATA_ROOT = Path('/content/microplastic_dataset')
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/Microplastic.v5i.yolov8')
yaml_files = list(DATA_ROOT.rglob('data.yaml')) if DATA_ROOT.exists() else []
if not yaml_files and DRIVE_DATA_ROOT.exists():
    yaml_files = list(DRIVE_DATA_ROOT.rglob('data.yaml'))
if not yaml_files:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError(f'Choose exactly one dataset ZIP; received: {zip_names}')
    zip_path = Path('/content') / zip_names[0]
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    root_resolved = DATA_ROOT.resolve()
    with ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (DATA_ROOT / member.filename).resolve()
            if target != root_resolved and root_resolved not in target.parents:
                raise ValueError(f'Unsafe path in archive: {member.filename}')
        archive.extractall(DATA_ROOT)
    yaml_files = list(DATA_ROOT.rglob('data.yaml'))
if len(yaml_files) != 1:
    raise RuntimeError(f'Expected exactly one data.yaml; found {len(yaml_files)}')

source_yaml = yaml_files[0]
dataset_root = source_yaml.parent
data = yaml.safe_load(source_yaml.read_text())
for split, folder in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    image_dir = dataset_root / folder / 'images'
    if not image_dir.exists():
        raise FileNotFoundError(f'Missing {split} images: {image_dir}')
    data[split] = str(image_dir)
data['path'] = str(dataset_root)
names = data['names']
class_names = [names.get(i, names.get(str(i))) if isinstance(names, dict) else names[i] for i in range(int(data['nc']))]
normalized_yaml = Path('/content/aquasentinel-accuracy-data.yaml')
normalized_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
print('Dataset root:', dataset_root)
print('Classes:', class_names)
print('Normalized config:', normalized_yaml)


## Optional supplemental fragment images (public dataset)

This cell downloads only the upstream **training split** (1,406 high-resolution images; around 0.9 GB) from the CC BY 4.0 Microplastic-Fragments-WWTP dataset on Hugging Face and maps its single annotated foreground category to your existing `Fragment` class. It adds these images only to model training. Your project's validation and test sets remain unchanged. Colab needs internet and enough free runtime storage; first download may take time. Set `USE_WWTP_SUPPLEMENT = False` to run without this source. The source contains fragment-specific labels and may include other particle shapes as unannotated background, so it is supplemental evidence, not a replacement for carefully labeled project data.

Source: [Dataset card](https://huggingface.co/datasets/andreusv/Microplastic-Fragments-WWTP-Dataset) (CC BY 4.0). The training script copies image pixels into Colab runtime files, not the source dataset into Git or Drive.


In [ ]:
from datasets import load_dataset
from PIL import Image

USE_WWTP_SUPPLEMENT = True
WWTP_DATASET_ID = 'andreusv/Microplastic-Fragments-WWTP-Dataset'
SUPPLEMENT_ROOT = Path('/content/microplastic_wwtp_supplement')
SUPPLEMENT_IMAGES = SUPPLEMENT_ROOT / 'images'
SUPPLEMENT_LABELS = SUPPLEMENT_ROOT / 'labels'
fragment_ids = [i for i, name in enumerate(class_names) if str(name).strip().lower() in {'fragment', 'fragments'}]
if USE_WWTP_SUPPLEMENT:
    if len(fragment_ids) != 1:
        raise ValueError(f'Need exactly one Fragment class in your data.yaml; found: {fragment_ids}, classes={class_names}')
    ready_marker = SUPPLEMENT_ROOT / 'READY.json'
    if not ready_marker.exists():
        SUPPLEMENT_IMAGES.mkdir(parents=True, exist_ok=True)
        SUPPLEMENT_LABELS.mkdir(parents=True, exist_ok=True)
        print('Downloading upstream train split (about 0.9 GB); this may take several minutes...')
        external_train = load_dataset(WWTP_DATASET_ID, split='train')
        for idx, row in enumerate(external_train):
            image = row['image'].convert('RGB')
            width, height = image.size
            stem = f'wwtp_{idx:05d}'
            image.save(SUPPLEMENT_IMAGES / f'{stem}.jpg', quality=95)
            objects = row['objects']
            boxes = objects.get('bbox', [])
            categories = objects.get('category_id', [])
            lines = []
            for box, category in zip(boxes, categories):
                if int(category) != 1:
                    continue
                x, y, box_w, box_h = map(float, box)
                x1, y1 = max(0.0, x), max(0.0, y)
                x2, y2 = min(float(width), x + box_w), min(float(height), y + box_h)
                if x2 <= x1 or y2 <= y1:
                    continue
                xc = ((x1 + x2) / 2) / width
                yc = ((y1 + y2) / 2) / height
                nw, nh = (x2 - x1) / width, (y2 - y1) / height
                lines.append(f'{fragment_ids[0]} {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}')
            (SUPPLEMENT_LABELS / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
            if (idx + 1) % 200 == 0:
                print(f'Converted {idx + 1}/{len(external_train)}')
        ready_marker.write_text(json.dumps({'source': WWTP_DATASET_ID, 'split': 'train', 'images': len(external_train), 'mapped_class': 'Fragment', 'license': 'CC BY 4.0'}, indent=2))
    data['train'] = [str(dataset_root / 'train' / 'images'), str(SUPPLEMENT_IMAGES)]
    print('Supplemental train images:', len(list(SUPPLEMENT_IMAGES.glob('*.jpg'))))
else:
    print('External supplemental data disabled.')
data['path'] = str(dataset_root)
normalized_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
print('Training image sources:', data['train'])
print('Validation and test remain the original project splits.')


In [ ]:
# Audit split labels and box geometry before retraining.
def audit_split(split_name, folder_name):
    image_dir = dataset_root / folder_name / 'images'
    label_dir = dataset_root / folder_name / 'labels'
    image_files = [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}]
    counts = Counter()
    areas = []
    missing = empty = malformed = invalid = 0
    for image_path in image_files:
        label_path = label_dir / (image_path.stem + '.txt')
        if not label_path.exists():
            missing += 1
            continue
        lines = [line.strip() for line in label_path.read_text(errors='replace').splitlines() if line.strip()]
        if not lines:
            empty += 1
        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                malformed += 1
                continue
            try:
                class_id = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:])
            except Exception:
                malformed += 1
                continue
            if class_id < 0 or class_id >= int(data['nc']) or not all(0 <= v <= 1 for v in (x, y, w, h)) or w <= 0 or h <= 0:
                invalid += 1
                continue
            counts[class_id] += 1
            areas.append(w * h)
    print(f'{split_name}: images={len(image_files)}, missing labels={missing}, empty labels={empty}, malformed rows={malformed}, invalid boxes={invalid}')
    print('  instances by class:', {class_names[i]: counts[i] for i in range(len(class_names))})
    if areas:
        import numpy as np
        print(f'  median normalized box area={float(np.median(areas)):.6f}; boxes below 0.1% image area={sum(a < 0.001 for a in areas)}/{len(areas)}')
    return counts
train_counts = audit_split('train', 'train')
val_counts = audit_split('validation', 'valid')

## Baseline validation

The baseline checkpoint is requested if it is not already in the Colab session or AquaSentinel Drive folder. This measures the existing model on validation data only.

In [ ]:
baseline_candidates = [Path('/content/best.pt'), Path('/content/drive/MyDrive/AquaSentinel-models/best.pt'), Path('/content/drive/MyDrive/AquaSentinel_Runs/train_v1-2/weights/best.pt'), Path('/content/drive/MyDrive/AquaSentinel_Runs/train_v1/weights/best.pt')]
BASELINE_PATH = next((p for p in baseline_candidates if p.exists()), None)
if BASELINE_PATH is None:
    uploaded = files.upload()
    pt_names = [name for name in uploaded if name.lower().endswith('.pt')]
    if len(pt_names) != 1:
        raise RuntimeError(f'Upload exactly one baseline .pt checkpoint; received: {pt_names}')
    BASELINE_PATH = Path('/content') / pt_names[0]
print('Baseline checkpoint:', BASELINE_PATH)
baseline = YOLO(str(BASELINE_PATH))
baseline_val = baseline.val(data=str(normalized_yaml), split='val', imgsz=640, device=0,
                            plots=True, project='/content/aquasentinel-evaluations',
                            name='baseline-val-640', exist_ok=True)
baseline_metrics = {k: float(v) for k, v in baseline_val.results_dict.items()}
print(json.dumps(baseline_metrics, indent=2))

## Train a separate candidate

This candidate uses YOLOv8s at 960 pixels. It gets a unique run name and does not overwrite the baseline. If CUDA runs out of memory, change batch from 4 to 2 and rerun this cell. Early stopping uses validation results.

In [ ]:
RUN_NAME = 'microplastic-yolov8s-960-' + datetime.now().strftime('%Y%m%d-%H%M')
RUN_DIR = Path('/content/aquasentinel-runs') / RUN_NAME
candidate_model = YOLO('yolov8s.pt')
candidate_model.train(data=str(normalized_yaml), epochs=100, imgsz=960, batch=4, device=0,
                      workers=2, project='/content/aquasentinel-runs', name=RUN_NAME,
                      exist_ok=False, plots=True, patience=20, save_period=10, seed=42)
CANDIDATE_PATH = RUN_DIR / 'weights' / 'best.pt'
if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(f'Candidate checkpoint missing: {CANDIDATE_PATH}')
print('Candidate checkpoint:', CANDIDATE_PATH)

In [ ]:
candidate_model = YOLO(str(CANDIDATE_PATH))
candidate_val = candidate_model.val(data=str(normalized_yaml), split='val', imgsz=960, device=0,
                                    plots=True, project='/content/aquasentinel-evaluations',
                                    name=RUN_NAME + '-val', exist_ok=True)
candidate_metrics = {k: float(v) for k, v in candidate_val.results_dict.items()}
print('Candidate validation metrics:')
print(json.dumps(candidate_metrics, indent=2))
print('Baseline validation metrics (640):')
print(json.dumps(baseline_metrics, indent=2))

## Compare confidence thresholds on validation data

Review validation metrics and the resulting plots. Higher confidence generally removes weak predictions but can also miss particles. Choose an operating threshold using validation data, not the test set.

In [ ]:
threshold_results = {}
for threshold in (0.15, 0.25, 0.35, 0.45):
    metrics = candidate_model.val(data=str(normalized_yaml), split='val', imgsz=960, device=0,
                                  conf=threshold, plots=True,
                                  project='/content/aquasentinel-evaluations',
                                  name=f'{RUN_NAME}-conf-{threshold:.2f}', exist_ok=True)
    threshold_results[str(threshold)] = {k: float(v) for k, v in metrics.results_dict.items()}
    print('confidence', threshold, json.dumps(threshold_results[str(threshold)], indent=2))

## Save the candidate and artifacts to Drive

The candidate is saved under a new filename. Keep the current best.pt until the candidate has been compared and evaluated. Do not replace it if validation results are worse.

In [ ]:
drive_out = Path('/content/drive/MyDrive/AquaSentinel-models') / RUN_NAME
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy2(CANDIDATE_PATH, drive_out / 'best-yolov8s-960.pt')
summary = {'run_name': RUN_NAME, 'baseline_validation_640': baseline_metrics,
           'candidate_validation_960': candidate_metrics,
           'candidate_confidence_sweep': threshold_results, 'classes': class_names}
(drive_out / 'validation-summary.json').write_text(json.dumps(summary, indent=2))
for source in [RUN_DIR / 'results.csv', RUN_DIR / 'results.png', RUN_DIR / 'confusion_matrix.png',
               RUN_DIR / 'PR_curve.png', RUN_DIR / 'F1_curve.png']:
    if source.exists():
        shutil.copy2(source, drive_out / source.name)
print('Saved candidate and available plots to:', drive_out)

## Final held-out test evaluation

Run this after choosing the candidate from validation results. The test split is for final reporting, not repeated tuning.

In [ ]:
final_test = candidate_model.val(data=str(normalized_yaml), split='test', imgsz=960, device=0,
                                plots=True, project='/content/aquasentinel-evaluations',
                                name=RUN_NAME + '-test-final', exist_ok=True)
final_test_metrics = {k: float(v) for k, v in final_test.results_dict.items()}
print('Final test metrics:', json.dumps(final_test_metrics, indent=2))
(drive_out / 'final-test-metrics.json').write_text(json.dumps(final_test_metrics, indent=2))
for source in [Path(final_test.save_dir) / 'confusion_matrix.png',
               Path(final_test.save_dir) / 'results.csv',
               Path(final_test.save_dir) / 'PR_curve.png',
               Path(final_test.save_dir) / 'F1_curve.png']:
    if source.exists():
        shutil.copy2(source, drive_out / source.name)
print('Final metrics saved to:', drive_out)